# Bayes and Naive Bayes: home assignment




In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
np.set_printoptions(precision=5, suppress=True)
print("Ready! Class order in manual examples: Insulator, Metal.")

Ready! Class order in manual examples: Insulator, Metal.


## Problem 1. Conditional probability

A new teaching dataset has rows [perovskite, not perovskite] and columns [oxide, not oxide]. Calculate the joint probability and both conditional probabilities.



In [2]:
counts = np.array([[4, 2], [3, 6]])
p_joint = 4/15  # TODO: P(perovskite and oxide)
p_p_given_o = 4/7  # TODO: P(perovskite | oxide)
p_o_given_p = 4/6  # TODO: P(oxide | perovskite)

if any(v is None for v in [p_joint, p_p_given_o, p_o_given_p]):
    print('Complete the three TODOs.')
else:
    assert np.allclose([p_joint, p_p_given_o, p_o_given_p], [4/15, 4/7, 4/6])
    print('Check passed:', p_joint, p_p_given_o, p_o_given_p)

Check passed: 0.26666666666666666 0.5714285714285714 0.6666666666666666


**Explain (1–2 sentences):** Why do the two conditional probabilities differ?

**Your answer:** …. The two conditional probabilities differ because they use different reference groups (denominators). P(perovskite|oxide)considers all oxide samples, whereas P(oxide|perovskite) considers all perovskite samples.

## Problem 2. One positive screening result

A stable material occurs with prior probability 0.10. A filter flags 80% of stable and 20% of unstable materials. Compute P(flag) and P(stable | flag).



In [3]:
prior, sensitivity, fpr = 0.10, 0.80, 0.20
p_flag = sensitivity * prior + fpr * (1 - prior)
posterior = (sensitivity * prior) / p_flag

if p_flag is None or posterior is None:
    print('Complete both TODOs.')
else:
    assert np.isclose(p_flag, 0.26)
    assert np.isclose(posterior*p_flag, sensitivity*prior)
    print('Check passed. Posterior:', posterior)

Check passed. Posterior: 0.30769230769230776


**Explain (1–2 sentences):** Why is the answer different from the sensitivity of 0.80?

**Your answer:** ….  Sensitivity of 0.80 means the filter correctly flags 80% of all stable materials, whereas P(stable| flag) is the probability that a flagged material is actually stable. Since stable materials are rare (only 10% of all materials) and unstable materials can also be flagged, the posterior probability is only about 30.77%.

## Problem 3. A second independent filter

Use the same rates as Problem 2. Write a reusable Bayes update function; apply it twice starting at 0.10. Assume the filters are independent conditional on the true class.


In [4]:
def update(prior, sensitivity, fpr):
    p_flag = sensitivity * prior + fpr * (1 - prior)
    posterior = (sensitivity * prior) / p_flag
    return posterior

first = update(0.10, 0.80, 0.20)
second = update(first, 0.80, 0.20)

if first is None or second is None:
    print('Complete the function and second update.')
else:
    assert np.isclose(first, 4/13)
    assert np.isclose(second, 0.64)
    print('Check passed:', first, second)

Check passed: 0.30769230769230776 0.64


**Explain (1–2 sentences):** Would applying the same deterministic filter twice justify this update?

**Your answer:** …. No, applying the same deterministic filter twice would not justify this update because the second result would not provide new independent evidence. The update is valid only if the two filter results are conditionally independent given the true class.

## Problem 4. Beta update

Start with Beta(3,7). Observe 4 stable candidates in 12 trials. Calculate the posterior parameters and mean.


In [5]:
a, b, k, n = 3, 7, 4, 12

a_post = a + k
b_post = b + (n - k)
post_mean = a_post / (a_post + b_post)

if any(v is None for v in [a_post, b_post, post_mean]):
    print('Complete all TODOs.')
else:
    assert a_post == 7 and b_post == 15
    assert np.isclose(post_mean, a_post/(a_post+b_post))
    print('Check passed:', a_post, b_post, post_mean)

Check passed: 7 15 0.3181818181818182


**Explain (1–2 sentences):** Compare the posterior mean with the prior mean and the observed success fraction.

**Your answer:** …. The prior mean is \(3/(3+7)=0.30\), while the observed success fraction is \(4/12 approx 0.3333\). The posterior mean is \(7/22 approx 0.3182\), which lies between them because it combines the prior belief with the observed data.

## Problem 5. Discrete classification
Classify a non-oxide with coordination number 6 and density below 5 g/cm³. Rows are [Insulator, Metal]; columns are [oxide, CN=6, low density]. Complement only the oxide column.


In [6]:
priors = np.array([2/3, 1/3])
base = np.array([[0.8, 0.5, 0.6], [0.3, 0.3, 0.4]])

observed = base.copy()
observed[:, 0] = 1 - observed[:, 0]

scores = priors * np.prod(observed, axis=1)
posterior = scores / scores.sum()

if any(v is None for v in [observed, scores, posterior]):
    print('Complete all TODOs.')
else:
    assert np.allclose(scores, [0.04, 0.028])
    assert np.allclose(posterior, scores/scores.sum())
    print('Check passed. Posterior [I,M]:', posterior)

Check passed. Posterior [I,M]: [0.58824 0.41176]


**Explain (1–2 sentences):** Which class wins, and why is the decision less decisive than for an oxide?

**Your answer:** …. The Insulator class wins with a posterior probability of approximately 58.82%, compared with 41.18% for Metal. The decision is less decisive than for an oxide because the observed non-oxide feature provides weaker evidence in favor of Insulator, so the posterior probabilities are closer together.

## Problem 6. Laplace smoothing
Of 20 metals, zero have a binary feature. Of 40 insulators, eight have it. Compute smoothed P(feature=yes | class) for each class.



In [7]:
p_yes_metal = (0 + 1) / (20 + 2)
p_yes_insulator = (8 + 1) / (40 + 2)

if p_yes_metal is None or p_yes_insulator is None:
    print('Complete both TODOs.')
else:
    assert np.allclose([p_yes_metal, p_yes_insulator], [1/22, 9/42])
    print('Check passed:', p_yes_metal, p_yes_insulator)

Check passed: 0.045454545454545456 0.21428571428571427


**Explain (1–2 sentences):** Why does an observed count of zero not establish physical impossibility?

**Your answer:** …. An observed count of zero only means that the feature was not observed in the available sample; it does not prove that the feature is physically impossible for that class. Laplace smoothing assigns a small, non-zero probability to account for unseen occurrences and avoid zero-probability predictions.

## Problem 7. Gaussian density

Write the Gaussian density formula. Evaluate x=0.7 for a metal atomic-size descriptor with mean 0.6 Å and standard deviation 0.2 Å. Also calculate the z-score.



In [8]:
def normal_density(x, mu, sigma):
    return (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(
        -0.5 * ((x - mu) / sigma) ** 2
    )

z = (0.7 - 0.6) / 0.2
density = normal_density(0.7, 0.6, 0.2)

if z is None or density is None:
    print('Complete both TODOs.')
else:
    assert np.isclose(z, 0.5)
    assert np.isclose(density, 1.7603266338)
    print('Check passed:', z, density)

Check passed: 0.4999999999999999 1.7603266338214976


**Explain (1–2 sentences):** Why is this density allowed to exceed one?

**Your answer:** … A probability density can exceed one because it represents probability per unit of measurement, not probability itself. The total area under the Gaussian curve must equal 1, while the probability over any interval is the area under the curve across that interval.

## Problem 8. Gaussian Naive Bayes

Classify x=[5.0,0.7,0.6]. Features are density, atomic size and electronegativity difference. Calculate the 2×3 density array, class scores, and normalized probabilities.



In [9]:
x = np.array([5.0, 0.7, 0.6])
means = np.array([[4, 1.3, 2.1], [4, 0.6, 0.4]])
stds = np.array([[2, 0.2, 0.4], [2, 0.2, 0.3]])
priors = np.array([2/3, 1/3])

# Calculate Gaussian density for each class and feature
heights = np.exp(-0.5 * ((x - means) / stds)**2) / (stds * np.sqrt(2 * np.pi))

# Calculate class scores
scores = priors * np.prod(heights, axis=1)

# Normalize scores to obtain posterior probabilities
posterior = scores / scores.sum()

if any(v is None for v in [heights, scores, posterior]):
    print('Complete all TODOs.')
else:
    assert np.shape(heights) == (2, 3)
    reference = np.exp(-0.5*((x-means)/stds)**2)/(stds*np.sqrt(2*np.pi))
    assert np.allclose(heights, reference)
    assert np.allclose(scores, priors*np.prod(reference, axis=1))
    assert np.allclose(posterior, scores/scores.sum())
    print('Check passed. Posterior [I,M]:', posterior)

Check passed. Posterior [I,M]: [0.00002 0.99998]


**Explain (1–2 sentences):** Why does density provide no discrimination here? Should the predicted probability be treated as experimentally established confidence?

**Your answer:** … Density provides no discrimination because both classes have the same mean (4) and standard deviation (2), so its Gaussian density is identical for both classes. The predicted probability should not be treated as experimentally established confidence, because it depends on the model assumptions and parameters and requires validation against experimental data.